# Comparing computed uvw against a real FITS-IDI file

`BL146_1.fits` is a VLBA observation (experiment BL146, 2007-08-23, correlated
at the VLBA hardware correlator) in FITS-IDI form. This notebook reads
everything the uvw computation needs from the file itself -- antenna ITRF
positions, station names, axis offsets, source coordinates, timestamps -- and
compares the file's stored uvw against this package's three methods:

* `calculate_uvw_astropy` -- pure geometric projection,
* `calculate_uvw_calc` -- CALC11 delays, geocenter reference (VLBI convention),
* `calculate_uvw_calc(mode="difxcalc11")` -- the embedded difxcalc11 pipeline.

Two conventions are tested along the way: the **baseline orientation**
(`uvw = P(antenna1) - P(antenna2)`, the archival convention this package and
MSv4 adopt) and the **aberration convention** (correlator delay-model uvw are
derived from aberrated delays; archived imaging uvw traditionally are not).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.time import Time

from radio_telescope_delay_model import calculate_uvw_astropy, calculate_uvw_calc

SPEED_OF_LIGHT = 299792458.0

hdus = fits.open("BL146_1.fits")
hdus.info()

## Metadata from the file

`ARRAY_GEOMETRY` carries the antenna ITRF positions (FRAME = GEOCENTRIC, array
centre at the origin), the two-letter VLBA station names (which the packaged
difxcalc11 catalogs know as aliases, so ocean loading applies), the mounts
(`MNTSTA = 0` = alt-az) and the axis offsets. `SOURCE` carries the J2000
coordinates. `UV_DATA` rows carry Julian `DATE` + day-fraction `TIME` (UTC),
the packed `BASELINE` code (256 a1 + a2, autocorrelations included) and
`UU-L/VV-L/WW-L` in light-seconds.

In [ ]:
array_geometry = hdus["ARRAY_GEOMETRY"]
antenna_position = np.asarray(array_geometry.data["STABXYZ"], dtype=np.float64)
station_name = [str(name) for name in array_geometry.data["ANNAME"]]
axis_offset = np.atleast_2d(
    np.asarray(array_geometry.data["STAXOF"], dtype=np.float64)
)[:, 0]
source_table = hdus["SOURCE"].data
uv = hdus["UV_DATA"].data

print("stations:", station_name)
print("mounts (0 = alt-az):", list(array_geometry.data["MNTSTA"]))
print("axis offsets (m):", axis_offset.round(3))
print("sources:", [str(s) for s in source_table["SOURCE"]])

antenna1 = (uv["BASELINE"] // 256).astype(int)
antenna2 = (uv["BASELINE"] % 256).astype(int)
cross = antenna1 != antenna2  # drop autocorrelations
file_uvw_all = (
    np.stack([uv["UU-L"], uv["VV-L"], uv["WW-L"]], axis=1).astype(np.float64)
    * SPEED_OF_LIGHT
)
print(
    f"\n{len(uv)} rows, {cross.sum()} cross-correlations; "
    f"max |uvw| = {np.abs(file_uvw_all[cross]).max():.0f} m"
)

## Compute uvw for every cross-correlation row

Rows are grouped by source; each group's unique epochs go through the three
methods, and model values are picked per row (handling the few rows whose
baseline code has the higher antenna first: for those the model baseline is
negated, since `uvw(i, j) = -uvw(j, i)`).

In [ ]:
results = {"astropy": [], "calc geometric": [], "difxcalc11": []}
file_rows = []

for source_id in sorted(set(uv["SOURCE"][cross].tolist())):
    rows = np.flatnonzero(cross & (uv["SOURCE"] == source_id))
    phase_center = np.deg2rad(
        [[source_table["RAEPO"][source_id - 1], source_table["DECEPO"][source_id - 1]]]
    )
    times = Time(
        np.asarray(uv["DATE"][rows]),
        np.asarray(uv["TIME"][rows]),
        format="jd",
        scale="utc",
    )
    unique_unix, inverse = np.unique(times.unix, return_inverse=True)
    epochs = Time(unique_unix, format="unix", scale="utc")

    uvw_calc, b1, b2 = calculate_uvw_calc(
        antenna_position,
        epochs,
        phase_center,
        reference_position=np.zeros(3),  # the VLBI convention
        station_name=station_name,
        axis_offset_metres=axis_offset,
    )
    uvw_difx, _, _ = calculate_uvw_calc(
        antenna_position,
        epochs,
        phase_center,
        mode="difxcalc11",
        station_name=station_name,
    )
    uvw_astropy, _, _ = calculate_uvw_astropy(antenna_position, epochs, phase_center)

    pair_index = {(int(x) + 1, int(y) + 1): k for k, (x, y) in enumerate(zip(b1, b2))}
    baseline_index, orientation = [], []
    for r in rows:
        i, j = antenna1[r], antenna2[r]
        if (i, j) in pair_index:
            baseline_index.append(pair_index[(i, j)])
            orientation.append(1.0)
        else:
            baseline_index.append(pair_index[(j, i)])
            orientation.append(-1.0)
    baseline_index = np.array(baseline_index)
    orientation = np.array(orientation)[:, None]

    file_rows.append(file_uvw_all[rows])
    for label, model in (
        ("astropy", uvw_astropy),
        ("calc geometric", uvw_calc),
        ("difxcalc11", uvw_difx),
    ):
        results[label].append(orientation * model[inverse, baseline_index])

file_uvw = np.concatenate(file_rows)
models = {label: np.concatenate(values) for label, values in results.items()}
print(
    f"compared {len(file_uvw)} rows across {len(set(uv['SOURCE'][cross].tolist()))} sources"
)

## Convention check and residuals

In [ ]:
print(
    f"{'method':16s} {'matched orientation':>22s} {'flipped orientation':>22s} "
    f"{'median |res|':>14s}"
)
for label, model in models.items():
    matched = np.abs(model - file_uvw).max()
    flipped = np.abs(-model - file_uvw).max()
    median = np.median(np.abs(model - file_uvw))
    print(f"{label:16s} {matched:18.2f} m {flipped:18.0f} m {median:12.3f} m")

The flipped-orientation column (off by the full uvw scale, ~1.6e7 m) confirms
the file carries the **archival convention `uvw = P(antenna1) - P(antenna2)`**
-- the convention this package produces directly.

The residual pattern identifies the file's aberration convention: the pure
geometric projection (astropy) matches at the metre level -- which is this
file's own precision floor, since `UU-L/VV-L/WW-L` are stored as float32
light-seconds (8.6e6 m x 1.2e-7 = ~1 m) -- while both CALC-derived methods
differ by hundreds of metres. That difference is the **annual aberration in
the uvw convention** (v/c ~ 1e-4 of the baseline): correlator delay-model uvw
(difxcalc's "EXACT" mode) are derived from aberrated delays, whereas this
2007 VLBA correlator FITS stores classic **unaberrated imaging uvw** (the
AIPS/UVFITS convention). The plot below shows the residuals following the
1e-4 x |B| aberration line.

In [ ]:
baseline_length = np.linalg.norm(file_uvw, axis=1)
figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(
    file_uvw[:, 0] / 1e6, file_uvw[:, 1] / 1e6, ".", ms=2, label="file", color="k"
)
axes[0].plot(
    models["astropy"][:, 0] / 1e6,
    models["astropy"][:, 1] / 1e6,
    ".",
    ms=1,
    label="computed (astropy)",
    color="tab:orange",
)
axes[0].set_xlabel("u (1000 km)")
axes[0].set_ylabel("v (1000 km)")
axes[0].set_title("uv coverage")
axes[0].legend()
axes[0].set_aspect("equal")

for label, color in (
    ("astropy", "tab:orange"),
    ("calc geometric", "tab:blue"),
    ("difxcalc11", "tab:green"),
):
    residual = np.linalg.norm(models[label] - file_uvw, axis=1)
    axes[1].plot(baseline_length / 1e6, residual, ".", ms=2, label=label, color=color)
grid = np.linspace(baseline_length.min(), baseline_length.max(), 50)
axes[1].plot(grid / 1e6, 1.0e-4 * grid, "k--", lw=1, label="1e-4 x |B| (aberration)")
axes[1].set_yscale("log")
axes[1].set_xlabel("|baseline| (1000 km)")
axes[1].set_ylabel("|uvw residual| (m)")
axes[1].set_title("residual vs baseline length")
axes[1].legend(fontsize=8)
plt.tight_layout()

## Conclusions

1. **Orientation**: the file's uvw are `P(antenna1) - P(antenna2)` (archival /
   MSv4 convention); the opposite orientation misses by the full uvw scale.
2. **Values**: computed uvw from nothing but the file's own metadata reproduce
   the stored uvw at the file's float32 precision floor (metre-level medians on
   up to 8 600 km baselines) when the matching aberration convention is used
   (the geometric projection, for this correlator vintage).
3. **Aberration convention**: the CALC-derived uvw (the difxcalc "EXACT"
   convention carried by DiFX `.im` models) differ from these archived imaging
   uvw by v/c ~ 1e-4 of the baseline -- up to ~800 m here. When comparing or
   mixing uvw sources at VLBI scales, the aberration convention matters as
   much as the sign convention.